# Desafío Individual: Sistema de Gestión de Obra Inteligente

## Contexto del Problema
Una empresa constructora está desarrollando una torre de gran altura y necesita automatizar dos procesos críticos para garantizar la seguridad y la eficiencia operativa:
* Evaluación de Riesgos en Obra (Lógica Deductiva): Determinar si es seguro continuar con las tareas de altura o excavación basándose en sensores climáticos y estructurales.
* Planificación de Maquinaria Pesada (Satisfacción de Restricciones): Asignar equipos (grúas, excavadoras) a zonas específicas del predio respetando límites de seguridad y espacio físico.

**Misión A:** Diagnóstico de Seguridad con expertaDebes programar un motor de inferencia que reciba datos de sensores y devuelva el nivel de riesgo del sitio. Este sistema actúa como un Cerebro Lógico para evitar accidentes.

Reglas a Implementar:
* **Riesgo Crítico** (Paro de Obra):  Si la velocidad del viento es $> 60\ km/h$ o si se detectan grietas en el suelo de fundación.
* **Riesgo Moderado** (Precaución): Si la velocidad del viento está entre $40\ km/h$ y $60\ km/h$ o si hay humedad extrema en zonas de excavación.
* **Bajo Riesgo** (Operación Normal): Si los vientos son $< 40\ km/h$ y no hay alertas estructurales activas.

**Consigna Técnica:** Utiliza el parámetro salience para asegurar que la regla de Riesgo Crítico se evalúe con la máxima prioridad ante cualquier otra condición.El sistema debe imprimir el diagnóstico final y la orden de seguridad correspondiente.

**Misión B:** Ubicación de Equipos con python-constraint

Debes encontrar la distribución óptima de 3 máquinas pesadas en 3 zonas de trabajo distintas. El sistema debe "podar" las opciones que violen las normativas de seguridad.

Restricciones (Reglas de Oro):
* **Grúa Torre:** Solo puede ubicarse en la Zona_Estable (debido a la necesidad de una base de hormigón reforzada).
* **Excavadora:** No puede ingresar a la Zona_Estrecha debido a sus dimensiones.
* **Hormigonera:** No puede estar en la misma zona que la Grúa Torre para evitar congestión de camiones.
* **Exclusividad:** Cada zona solo puede albergar una máquina a la vez para evitar colisiones.Consigna Técnica:Define las variables (Máquinas) y el dominio (Zonas de la obra).

Aplica las funciones de restricción para que el motor de búsqueda encuentre la única configuración válida.

## Parámetros de Entrega y Evaluación
El entregable debe cumplir con lo siguiente:

* **Justificación Funcional:** Debes explicar en celdas de texto por qué el modelo de grafos y árboles es superior a una simple lista de if/else para este problema.
* **Documentación:** El código debe estar respaldado por una explicación de cómo opera el motor de inferencia en cada caso.

*Tener en cuenta que se evalua proceso y no resultado. Acordarse de justificar las elecciones en cada caso.*

# RESOLUCION

🏗️ Justificación: ¿Por qué un Sistema Experto?
No usamos simples if/else porque en una obra real:

La seguridad no puede esperar: Los if/else leen una línea a la vez. El Sistema Experto evalúa todo en paralelo y, gracias al Salience, la regla de "Riesgo Crítico" siempre salta al principio de la fila, sin importar qué más esté pasando.

Es modular: Si mañana quieres agregar un sensor de "sismos", solo añades una regla nueva. No tienes que desarmar y volver a armar todo el código.

Inteligencia Separada: Los datos (sensores) están separados de la lógica (reglas). Esto permite que el sistema sea fácil de entender y actualizar por ingenieros, no solo por programadores.

📚 Operación del Motor (Ciclo de Vida)
El sistema funciona en un bucle de 3 pasos rápidos:

Match (Emparejar): El motor mira los sensores y busca qué reglas coinciden.

Conflict (Priorizar): Si coinciden varias reglas (ej: hay viento y también humedad), el motor usa el Salience para elegir la más importante.

Act (Actuar): Ejecuta la orden (ej: "Paro de Obra").

# A

In [ ]:
# 1. Instalamos la librería
!pip install experta

# 2. PARCHE DE COMPATIBILIDAD (Ejecuta esto antes del import)
import collections
if not hasattr(collections, 'Mapping'):
    import collections.abc
    collections.Mapping = collections.abc.Mapping

# 3. Ahora sí, importamos experta
from experta import *

In [ ]:
# 1. Instalación y Carga de Librerías
try:
    from experta import *
except ImportError:
    !pip install experta
    from experta import *

# 2. Definición de la estructura de datos (Hechos)
class SensorData(Fact):
    """
    Representa los datos crudos capturados por los sensores en la torre.
    Campos: viento (int), grietas (bool), humedad_extrema (bool).
    """
    pass

# 3. Construcción del Motor de Inferencia
class EvaluadorSeguridad(KnowledgeEngine):

    # REGLA DE PRIORIDAD MÁXIMA: RIESGO CRÍTICO
    @Rule(
        # Dispara si el viento > 60 O si hay grietas detectadas
        (SensorData(viento=P(lambda x: x > 60))) |
        (SensorData(grietas=True)),
        salience=10 # Asegura que esta regla sea la primera en ejecutarse
    )
    def alerta_critica(self):
        self.imprimir_resultado("CRÍTICO 🚨", "PARO DE OBRA INMEDIATO y evacuación.")

    # REGLA DE PRIORIDAD MEDIA: RIESGO MODERADO
    @Rule(
        (SensorData(viento=P(lambda x: 40 <= x <= 60))) |
        (SensorData(humedad_extrema=True)),
        salience=5
    )
    def alerta_moderada(self):
        self.imprimir_resultado("MODERADO ⚠️", "PRECAUCIÓN. Monitoreo constante de suelos.")

    # REGLA DE OPERACIÓN NORMAL: BAJO RIESGO
    @Rule(
        SensorData(viento=P(lambda x: x < 40),
                   grietas=False,
                   humedad_extrema=False),
        salience=1
    )
    def operacion_normal(self):
        self.imprimir_resultado("BAJO ✅", "OPERACIÓN NORMAL. Continuar según cronograma.")

    def imprimir_resultado(self, nivel, orden):
        print(f"{'='*40}")
        print(f"DIAGNÓSTICO DE SEGURIDAD: {nivel}")
        print(f"ORDEN DE ACCIÓN: {orden}")
        print(f"{'='*40}\n")

# 4. Pruebas de Proceso (Simulación)
engine = EvaluadorSeguridad()
engine.reset()

# Caso de prueba: Sensor detecta una situación mixta (viento alto y humedad)
print("SIMULANDO: Viento 65km/h y Humedad Extrema detectada...")
engine.declare(SensorData(viento=65, grietas=False, humedad_extrema=True))
engine.run()

SIMULANDO: Viento 65km/h y Humedad Extrema detectada...
DIAGNÓSTICO DE SEGURIDAD: CRÍTICO 🚨
ORDEN DE ACCIÓN: PARO DE OBRA INMEDIATO y evacuación.

DIAGNÓSTICO DE SEGURIDAD: MODERADO ⚠️
ORDEN DE ACCIÓN: PRECAUCIÓN. Monitoreo constante de suelos.



🏗️ Justificación Funcional: ¿Por qué CSP?

Eficiencia en la búsqueda: No prueba todas las combinaciones al azar. Si una máquina no cabe en una zona, el sistema "poda" esa rama del árbol de decisiones inmediatamente.

Seguridad Garantizada: El motor de búsqueda solo entrega resultados que cumplen todas las reglas (Restricciones de Oro). Si no hay solución segura, el sistema avisa en lugar de proponer algo peligroso.

# B

In [ ]:
!pip install python-constraint

  Preparing metadata (setup.py) ... done
  Created wheel for python-constraint: filename=python_constraint-1.4.0-py2.py3-none-any.whl size=24061 sha256=972e206b075d50427891decf5ba4369e652a0e563de72a72ecab78e7eaf18067
  Stored in directory: /root/.cache/pip/wheels/c1/d2/3d/082849b61a9c6de02d4a7c8a402c224640f08d8a971307b92b
Successfully built python-constraint


In [ ]:
from constraint import *

# 1. Crear el Problema
problem = Problem()

# 2. Definir Variables (Máquinas) y Dominios (Zonas)
maquinas = ["Grua_Torre", "Excavadora", "Hormigonera"]
zonas = ["Zona_Estable", "Zona_Estrecha", "Zona_Libre"]

# Añadimos las variables al sistema
problem.addVariables(maquinas, zonas)

# 3. Aplicar las "Reglas de Oro" (Restricciones)

# REGLA 1: La Grúa Torre DEBE estar en la Zona_Estable (base reforzada)
problem.addConstraint(lambda g: g == "Zona_Estable", ["Grua_Torre"])

# REGLA 2: La Excavadora NO puede estar en la Zona_Estrecha
problem.addConstraint(lambda e: e != "Zona_Estrecha", ["Excavadora"])

# REGLA 3: La Hormigonera NO puede estar donde está la Grúa (evitar congestión)
problem.addConstraint(lambda h, g: h != g, ["Hormigonera", "Grua_Torre"])

# REGLA 4: Exclusividad (Una máquina por zona para evitar choques)
problem.addConstraint(AllDifferentConstraint())

# 4. Obtener la solución válida
soluciones = problem.getSolutions()

print("--- PLANIFICACIÓN DE MAQUINARIA PESADA ---")
if soluciones:
    for sol in soluciones:
        print(f"Configuración Segura Encontrada:")
        for maq, zona in sol.items():
            print(f" -> {maq}: {zona}")
else:
    print("ALERTA: No existe una configuración que cumpla todas las normas de seguridad.")

--- PLANIFICACIÓN DE MAQUINARIA PESADA ---
Configuración Segura Encontrada:
 -> Grua_Torre: Zona_Estable
 -> Hormigonera: Zona_Estrecha
 -> Excavadora: Zona_Libre


Variables y Dominios: Las variables son los objetos que queremos ubicar (Máquinas) y el dominio son las opciones disponibles (Zonas).

Filtrado por Restricciones: El motor de python-constraint utiliza un algoritmo de búsqueda. Cada vez que intenta asignar una zona a una máquina, revisa las "Reglas de Oro". Si la regla dice "Falso", descarta ese camino. A esto se le llama "poda del árbol de búsqueda